<a id="unit-u.d7fa8053df0142548cf20306a1a8433e"></a>

<div>
<div>КЕҢЕЙТУ · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
<h1>2–3-модульдердің кеңейтілген зертханалық жұмысы: пакеттер, бөлімдер және Profile</h1>
<p>Мақсатты нұсқа — Doris 4.1.3 · Тәуелсіз ext_* зертханалық кестелері</p>
</div>

[2-модуль](../module02-architecture/course.md) · [3-модуль](../module03-table-design/course.md) · [Курстың README файлы](../../README.md)

Оны 2 және 3-зертханалық жұмыстардан кейін курстың бір контейнерлі сынақ ортасында орындаңыз. Ол WWI деректері мен бизнес метрикаларынан бөлек сақталатын 100,000 жолдық бекітілген оқу деректерін жасайды және тек `ext_small_batches`, `ext_large_batch` пен `ext_partitioned` кестелерін қайта жасайды. Бір уақытта тек бір ядрода орындаңыз.
25–35 минут бөліңіз. Орындамас бұрын SQL-ді оқыңыз. Қадам сәтсіз аяқталса, оның шығысын сақтап, тек осы үш кестені қалпына келтіретін баптау ұяшығынан қайта бастаңыз. Баптау ұяшығы сынақ ортасының BE-сін көрсетеді: `Alive` true және `Version` `doris-4.1.3-rc02-7126cf65d96`.

Үш кестеде бірдей деректер болғанда, таңдалған күнде жалпы сомасы 100,000.00 болатын 10,000 жол болғанда және олардың жоспарларын, пакеттері мен нұсқаларын, Profile-дарын салыстырғанда жұмыс аяқталады. Шағын пакеттер міндетті түрде нұсқалар кезегін қалдырмайды және бір уақыт өлшемі жылдамдаудың дәлелі емес.


In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql, show_records
lab = connect_sandbox()
lab.sql("SHOW BACKENDS", title="The course sandbox's BE", columns=["Host", "Alive", "Version"]);

<a id="unit-u.9e5a3436d175409f8331a12ccefce39c"></a>

<a id="1-fixed-input-and-table-structure"></a>

## 1. Тұрақты кіріс және кесте құрылымы

Әр 10,000 жол бір күнді білдіреді, сома 10.00 болып бекітілген. Алғашқы екі кестенің схемалары бірдей, тек жазу пакеттерінің өлшемі өзгеше.
Үшіншісі сол кілт бағандары мен бакеттерге бөлуді сақтап, тек күндік бөлімдер қосады; осылайша сұрыптау кілттері мен бакет саны бір мезгілде өзгермейді.


In [ ]:
from time import perf_counter
fields = "order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(12,2) NOT NULL"
partitions = ",".join(f"PARTITION p{day:02} VALUES [('2026-01-{day:02}'), ('2026-01-{day+1:02}'))" for day in range(1,11))
for table in ("ext_small_batches", "ext_large_batch", "ext_partitioned"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    partition = f"PARTITION BY RANGE(order_date) ({partitions})" if table == "ext_partitioned" else ""
    ddl = f'CREATE TABLE {table} ({fields}) DUPLICATE KEY(order_date,order_id) {partition} DISTRIBUTED BY HASH(order_id) BUCKETS 4 PROPERTIES("replication_num"="1")'
    show_sql("Physical design", ddl)
    lab.execute(ddl)
source = """SELECT DATE_ADD(CAST('2026-01-01' AS DATE), INTERVAL CAST(FLOOR(number / 10000) AS INT) DAY),
number, CAST(10 AS DECIMAL(12,2)) FROM numbers("number"="100000")"""
write_measurements = []
previous_group = lab.query("SELECT @@group_commit")[0][0]
try:
    lab.execute("SET group_commit = 'off_mode'")
    for table, batch_size in (("ext_small_batches",1000), ("ext_large_batch",100000), ("ext_partitioned",100000)):
        start = perf_counter()
        for offset in range(0,100000,batch_size):
            lab.execute(f"INSERT INTO {table} {source} WHERE number >= {offset} AND number < {offset+batch_size}")
        write_measurements.append({"Table": table, "Submitted batches": 100000//batch_size,
                                   "Write seconds": round(perf_counter()-start,3)})
        expect(lab.query(f"SELECT COUNT(*), SUM(amount),COUNT(DISTINCT order_id) FROM {table}"), [(100000,"1000000.00",100000)])
        lab.sql(f"SHOW TABLETS FROM {table}", title="Tablet status after writes (not fixed expected values)")
finally:
    lab.execute("SET group_commit = %s", (previous_group,))
show_records("Write measurements", write_measurements)
for table in ("ext_small_batches", "ext_partitioned"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} a FULL OUTER JOIN ext_large_batch b ON a.order_id=b.order_id WHERE a.order_id IS NULL OR b.order_id IS NULL OR a.order_date<>b.order_date OR a.amount<>b.amount"), [(0,)])

<a id="unit-u.00a231a21bad4b428cb8ef07e5f017e2"></a>

<a id="2-results-pruning-and-profile"></a>

## 2. Нәтижелер, артық деректерді оқудан алып тастау және Profile

Бөлімделмеген және бөлімделген кестелерде бірдей предикатты орындаңыз. Бір рет қыздыру орындауын жасаңыз, содан кейін бес айналымда екі кестені кезектестіріп, әр ұзақтықты сақтаңыз; бұл параллель бенчмарк те, P95 те емес.
Бөлімдер/Tablet-терді оқудан алып тастауды тексеру үшін `EXPLAIN`, ал іс жүзінде сканерленген жолдар мен оператор уақыттарын тексеру үшін Profile қолданыңыз. Бірдей сұрау нәтижесі бірдей сканерлеу көлемін білдірмейді.
Rowset-терді 2-курстағы `SHOW TABLET → DetailCmd → CompactionStatus` жолы арқылы тексеруді жалғастыра аласыз; Compaction-ды қолмен іске қоспаңыз.


In [ ]:
predicate = "order_date = '2026-01-03'"
query_measurements = []
from doris_labkit.profiles import last_query_id
previous_profile = lab.query("SELECT @@enable_profile, @@enable_sql_cache, @@enable_condition_cache")[0]
try:
    lab.execute("SET enable_profile = true, enable_sql_cache = false, enable_condition_cache = false")
    for table in ("ext_large_batch", "ext_partitioned"):
        lab.sql(f"EXPLAIN SELECT SUM(amount) FROM {table} WHERE {predicate}", title=table + " plan")
        expect(lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}"), [(10000,"100000.00")])
    for iteration in range(5):
        for table in ("ext_large_batch", "ext_partitioned"):
            start = perf_counter()
            result = lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}")
            elapsed_ms = round((perf_counter()-start)*1000,3)
            query_id = last_query_id(lab.connection)  # Before any other SQL can replace the ID.
            expect(result, [(10000,"100000.00")])
            query_measurements.append({"Iteration": iteration+1, "Table": table,
                                       "Query ms": elapsed_ms, "query_id": query_id,
                                       "profile_url": lab.profile_url(query_id)})
    show_records("Query timing comparison", query_measurements)
    lab.sql("SHOW QUERY PROFILE", title="Find this lab's Profile by SQL, database name, and time")
finally:
    lab.execute("SET enable_profile = %s, enable_sql_cache = %s, enable_condition_cache = %s", previous_profile)

<a id="unit-u.1d4a0734ef6746dc99912eab7396b9a0"></a>

<a id="3-independent-explanation-and-troubleshooting"></a>

## 3. Өз бетінше түсіндіру және ақауларды анықтау

Әр уақыт жолында нақты `query_id` және FE сілтемесі сақталады. Сол ID-лердің екеуін ашып, SQL мен дерекқорын тексеріңіз, содан кейін `ScanRows`, `ScanBytes` және Profile кэштелгенін салыстырыңыз. Секундомер ID алынғанға дейін тоқтайды, сондықтан ол Profile іздеуді емес, деректер сұрауын өлшейді. Бұл қадамда SQL және шарт кэштері өшіріліп, кейін қалпына келтіріледі.

Екі жоспардағы partitions/tablets өрістерін жазып, Profile ішінен сәйкес сканерлеу операторларын табыңыз; ұзақтық азаймаса, нәтижелерді өзгертпей, оны дерек көлемі, кэштеу және сұраудың тұрақты үстеме шығындары арқылы түсіндіріңіз.
Шағын пакетті 5,000 жолға өзгертіп, жұмысты қайта орындаңыз; нұсқа сандары пакет санына дәл тең болуын талап етпей, нақты пакет сандарын және VersionCount мәндерін салыстырыңыз.
Profile тізімінде жазба әлі болмаса, кейінірек `SHOW QUERY PROFILE` қайта орындаңыз; өрістердің толық сипаттамасын [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/) бөлімінен қараңыз.
finally блогы сеанс параметрін қалпына келтіреді; соңында деректерді жоймай, ақауларды анықтау үшін үш кестені де сақтаңыз.


In [ ]:
lab.close()